# 0.0 Preparação

## 0.1 Imports

In [ ]:
from pathlib import Path
from pickle import dump
from random import sample

import inflection
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from boruta import BorutaPy
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, RobustScaler
from tabulate import tabulate
from xgboost import XGBRegressor

from utils.cramer_v import cramer_v
from utils.cross_validation import cross_validation
from utils.ml_error_regression import ml_error_regression

## 0.2 Carregando Dados

In [ ]:
df_sales_raw = pd.read_csv(
    r"C:\Users\Admin\.cache\kagglehub\competitions\rossmann-store-sales\train.csv", low_memory=False
)
df_store_raw = pd.read_csv(
    r"C:\Users\Admin\.cache\kagglehub\competitions\rossmann-store-sales\store.csv", low_memory=False
)

df_raw = pd.merge(df_sales_raw, df_store_raw, how="left", on="Store")

# 1.0 Passo 1 - Entendimento dos Dados

In [ ]:
df1 = df_raw.copy()

## 1.1 Renomeando As Colunas

In [ ]:
cols_old = [
    "Store",
    "DayOfWeek",
    "Date",
    "Sales",
    "Customers",
    "Open",
    "Promo",
    "StateHoliday",
    "SchoolHoliday",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
]

snakecase = lambda x: inflection.underscore(x)

cols_new = list(map(snakecase, cols_old))

df1.columns = cols_new

## 1.2 Dimensão Dos Dados

In [ ]:
print(f"Numero de linhas: {df1.shape[0]}")
print(f"Numero de colunas: {df1.shape[1]}")

## 1.3 Tipo De Dados

In [ ]:
df1["date"] = pd.to_datetime(df1["date"])
df1.dtypes

## 1.4 Analisar os NA

In [ ]:
df1.isna().sum()

## 1.5 Filtrando os NA

In [ ]:
month_map = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec",
}

df1["competition_distance"] = df1["competition_distance"].fillna(200000)

df1["competition_open_since_month"] = df1["competition_open_since_month"].fillna(
    df1["date"].dt.month
)
df1["competition_open_since_year"] = df1["competition_open_since_year"].fillna(df1["date"].dt.year)

df1["promo2_since_week"] = df1["promo2_since_week"].fillna(df1["date"].dt.isocalendar().week)
df1["promo2_since_year"] = df1["promo2_since_year"].fillna(df1["date"].dt.year)

df1["promo_interval"] = df1["promo_interval"].fillna(0)
df1["month_map"] = df1["date"].dt.month.map(month_map)

df1["is_promo"] = df1[["promo_interval", "month_map"]].apply(
    lambda x: (
        0
        if x["promo_interval"] == 0
        else (1 if x["month_map"] in str(x["promo_interval"]).split(",") else 0)
    ),
    axis=1,
)

## 1.6 Mudando O Tipo Dos Dados

In [ ]:
df1["competition_open_since_month"] = df1["competition_open_since_month"].astype(int)
df1["competition_open_since_year"] = df1["competition_open_since_year"].astype(int)
df1["promo2_since_week"] = df1["promo2_since_week"].astype(int)
df1["promo2_since_year"] = df1["promo2_since_year"].astype(int)

## 1.7 Descrição Estatistica

In [ ]:
num_attributes = df1.select_dtypes(include=["int64", "float64"])
cat_attributes = df1.select_dtypes(exclude=["int64", "float64", "datetime64"])

### 1.7.1 Atributos Numericos

In [ ]:
# Tendencias centrais - Media e Mediana
m1 = pd.DataFrame(num_attributes.apply(np.mean)).T
m2 = pd.DataFrame(num_attributes.apply(np.median)).T

# Dispersão - std, min, max, range, skew e kurtosis
d1 = pd.DataFrame(num_attributes.apply(np.std)).T
d2 = pd.DataFrame(num_attributes.apply(np.min)).T
d3 = pd.DataFrame(num_attributes.apply(np.max)).T
d4 = pd.DataFrame(num_attributes.apply(lambda x: x.max() - x.min())).T
d5 = pd.DataFrame(num_attributes.apply(lambda x: x.skew())).T
d6 = pd.DataFrame(num_attributes.apply(lambda x: x.kurtosis())).T

# Concatena
m = pd.concat([d2, d3, d4, m1, m2, d1, d5, d6]).T
m.columns = ["min", "max", "range", "mean", "median", "std", "skew", "kurtosis"]
m

### 1.7.2 Atributos Categoricos

In [ ]:
cat_attributes.apply(lambda x: x.unique().shape[0])

In [ ]:
aux1 = df1[(df1["state_holiday"] != "0") & (df1["sales"] > 0)]

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("State Holiday vs Sales", "Store Type vs Sales", "Assortment vs Sales"),
)

fig1 = px.box(aux1, x="state_holiday", y="sales", color="state_holiday")
fig2 = px.box(aux1, x="store_type", y="sales", color="store_type")
fig3 = px.box(aux1, x="assortment", y="sales", color="assortment")

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

for trace in fig3.data:
    fig.add_trace(trace, row=1, col=3)

fig.update_layout(
    height=600,  # Altura do gráfico em pixels
    autosize=True,  # Preenche a largura total disponível
    template="plotly_white",  # Tema limpo com alto contraste
    showlegend=False,  # Opcional: oculta a legenda lateral para dar mais espaço
    title_text="Análise de Vendas por Atributos Categóricos",
)

fig.show()

# 2.0 Passo 2 - Engenharia de Recursos

In [ ]:
df2 = df1.copy()

## 2.1 Criação Das Hipoteses

### 2.1.1 Hipoteses Loja

**1.** Lojas com maior quadro de funcionarios deveriam vender mais.

**2.** Lojas com maior estoque deveriam vender mais.

**3.** Lojas com maior porte deveriam vender mais.

**4.** Lojas com maior porte deveriam vender menos

**5.** Lojas com maior sortimento deveriam vender mais.

**6.** Lojas com competidores a mais tempo deveriam vender mais.

### 2.1.2 Hipoteses Produtos

**1.** Lojas que investem mais em marketing deveriam vender mais.

**2.** Lojas que expoe mais os produtos nas vitrines deveriam vender mais.

**3.** Lojas que tem preços menores nos produtos deveriam vender mais.

**4.** Lojas com promoções mais agressivas (descontos maiores), deveriam vender mais.

**5.** Lojas com promoções ativas por mais tempo deveriam vender mais.

**6.** Lojas com mais dias de promoções deveriam vender mais.

**7.** Lojas com mais promoções consecutivas deveriam vender mais.

### 2.1.3 Hipoteses Tempo

**1.** Lojas que tem mais feriados deveriam vender menos.

**2.** Lojas que abrem nos primeiros 6 meses deveriam vender mais.

**3.** Lojas que abrem nos finais de semana deveriam vender mais.

**4.** Lojas deveriam vender mais depois do dia 10 de cada mês.

**5.** Lojas deveriam vender menos aos finais de semana.

**6.** Lojas deveriam vender durante os feriados escolares.

## 2.2 Lista Final de Hipóteses

**1.** Lojas com maior sortimentos deveriam vender mais.

**2.** Lojas com competidores mais proximos deveriam vender menos.

**3.** Lojas com competidores à mais deveriam vender mais.

**4.** Lojas com promoções ativas por mais tempo deveriam vender mais.

**5.** Lojas com mais dias de promoção deveriam vender mais.

**6.** Lojas com mais promoções consecutivas deveriam vender mais.

**7.** Lojas abertas durante o feriado de Natal deveriam vender mais.

**8.** Lojas deveriam vender mais ao longo dos anos.

**9.** Lojas deveriam vender mais no segundo semestre do ano.

**10.** Lojas deveriam vender mais depois do dia 10 de cada mês.

**11.** Lojas deveriam vender menos aos finais de semana.

**12.** Lojas deveriam vender menos durante os feriados escolares.

## 2.3 Engenharia de Recursos

In [ ]:
# Colunas de tempo
df2["year"] = df2["date"].dt.year
df2["month"] = df2["date"].dt.month
df2["day"] = df2["date"].dt.day
df2["week_of_year"] = df2["date"].dt.isocalendar().week

df2["year_week"] = df2["date"].dt.strftime("%Y-%W")

# Colunas de Competição por tempo
year = df2["competition_open_since_year"].astype(int).astype(str)
month = df2["competition_open_since_month"].astype(int).astype(str)

df2["competition_since"] = pd.to_datetime(year + "-" + month + "-01", errors="coerce")
df2["competition_time_month"] = ((df2["date"] - df2["competition_since"]).dt.days / 30).astype(int)

# Colunas de Promoção
df2["date"] = df2["date"].dt.tz_localize(None)
promo_date_str = (
    df2["promo2_since_year"].astype(str) + "-" + df2["promo2_since_week"].astype(str) + "-1"
)

df2["promo2_since"] = pd.to_datetime(promo_date_str, format="%Y-%W-%w") - pd.Timedelta(days=7)
df2["promo_time_week"] = ((df2["date"] - df2["promo2_since"]).dt.days / 7).astype(int)

# Sortimento
df2["assortment"] = df2["assortment"].apply(
    lambda x: "basic" if x == "a" else "extra" if x == "b" else "extended"
)

# Feriados nacionais
# Mapeamento dos feriados
HOLIDAY_MAP = {"a": "public_holiday", "b": "easter_holiday", "c": "christmas"}
df2["state_holiday"] = df2["state_holiday"].map(HOLIDAY_MAP).fillna("regular_day")

# 3.0 Passo 3 - Filtragem de Variáveis

In [ ]:
df3 = df2.copy()

## 3.1 Filtragem Das Linhas

In [ ]:
df3 = df3[(df3["open"] != 0) & (df3["sales"] > 0)]

## 3.2 Seleção Das Colunas

In [ ]:
cols_drop = ["customers", "open", "promo_interval", "month_map"]
df3 = df3.drop(cols_drop, axis=1)

# 4.0 Passo 04 - Analise Exploratoria Dos Dados

In [ ]:
df4 = df3.copy()

## 4.1 - Analise Univariada

### 4.1.1 Variaveis de Resposta

In [ ]:
fig = px.histogram(
    df4, x="sales", title="Distribuição de Vendas (sales)", nbins=50, template="plotly_white"
)

fig.update_layout(xaxis_title="sales", yaxis_title="Count", height=500)

fig.show()

### 4.1.2 Variaveis Numericas

In [ ]:
num_attributes.hist(bins=25, figsize=(20, 12))

plt.tight_layout()
plt.show()

### 4.1.3 Variaveis Categoricas

In [ ]:
df4["state_holiday"].drop_duplicates()

In [ ]:
import plotly.express as px
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

# Preparação dos dados
aux_holiday = df4[df4["state_holiday"] != "regular_day"]

# Criação do Grid 2x2
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Contagem por Feriado",
        "Densidade de Vendas por Feriado",
        "Contagem por Tipo de Loja",
        "Densidade de Vendas por Tipo de Loja",
    ),
)

# --- Subplot 1: Countplot State Holiday ---
fig1 = px.histogram(aux_holiday, x="state_holiday", color="state_holiday")
for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

# --- Subplot 2: KDE Sales por State Holiday ---
holidays = ["public_holiday", "easter_holiday", "christmas"]
sales_by_holiday = [df4[df4["state_holiday"] == h]["sales"].dropna() for h in holidays]

fig2 = ff.create_distplot(sales_by_holiday, holidays, show_hist=False, show_rug=False)
for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

# --- Subplot 3: Countplot Store Type ---
fig3 = px.histogram(df4, x="store_type", color="store_type")
for trace in fig3.data:
    fig.add_trace(trace, row=2, col=1)

# --- Subplot 4: KDE Sales por Store Type ---
store_types = sorted(df4["store_type"].unique())
sales_by_store = [df4[df4["store_type"] == st]["sales"].dropna() for st in store_types]

fig4 = ff.create_distplot(
    sales_by_store, [str(st) for st in store_types], show_hist=False, show_rug=False
)
for trace in fig4.data:
    fig.add_trace(trace, row=2, col=2)

# Layout e exibição
fig.update_layout(
    height=800,
    autosize=True,
    template="plotly_white",
    title_text="Análise Exploratória de Dados (EDA) - Categorias vs Vendas",
    showlegend=True,
)

fig.show()

## 4.2 - Analise Bivariada

#### H1. Lojas com maior sortimentos deveriam vender mais.

**FALSO**: Lojas com MAIOR SORTIMENTO VENDEM MENOS.

In [ ]:
aux1 = df4[["assortment", "sales"]].groupby("assortment").sum().reset_index()

fig = px.bar(
    aux1,
    x="assortment",
    y="sales",
    color="assortment",
    title="Total de Vendas por Assortment",
    text_auto=".2s",
    template="plotly_white",
)

fig.update_layout(xaxis_title="Assortment", yaxis_title="Total Sales", height=500, showlegend=False)

fig.show()

In [ ]:
aux2 = (
    df4[["year_week", "assortment", "sales"]]
    .groupby(["year_week", "assortment"])
    .sum()
    .reset_index()
)

aux2.pivot(index="year_week", columns="assortment", values="sales").plot()

In [ ]:
aux3 = aux2[aux2["assortment"] == "extra"]

aux3.pivot(index="year_week", columns="assortment", values="sales").plot()

#### H2. Lojas com competidores mais próximos deveriam vender menos.

**Falso**: Lojas com competidores mais proximos vendem MAIS.

In [ ]:
aux1 = df4[["competition_distance", "sales"]].groupby("competition_distance").sum().reset_index()

bins = list(np.arange(0, 20000, 1000))
aux1["competition_distance_binned"] = pd.cut(aux1["competition_distance"], bins=bins)
aux2 = (
    aux1[["competition_distance_binned", "sales"]]
    .groupby("competition_distance_binned")
    .sum()
    .reset_index()
)

aux2["competition_distance_binned"] = aux2["competition_distance_binned"].astype(str)

corr_matrix = aux1[["competition_distance", "sales"]].corr(method="pearson")

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Scatterplot: Distância vs Vendas",
        "Vendas por Faixa de Distância",
        "Matriz de Correlação (Pearson)",
    ),
)

# Subplot 1: Scatterplot
fig_scatter = px.scatter(aux1, x="competition_distance", y="sales")
for trace in fig_scatter.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot 2: Barplot
fig_bar = px.bar(aux2, x="competition_distance_binned", y="sales", text_auto=".2s")
for trace in fig_bar.data:
    fig.add_trace(trace, row=1, col=2)

# Subplot 3: Heatmap (Sem problemas de bordas cortadas)
fig_heatmap = px.imshow(corr_matrix, text_auto=".2f", color_continuous_scale="RdBu_r")
for trace in fig_heatmap.data:
    fig.add_trace(trace, row=1, col=3)

# Ajustes Finais de Layout
fig.update_layout(
    height=500,
    width=1600,
    template="plotly_white",
    showlegend=False,
    title_text="Análise de Distância da Concorrência vs Vendas",
)

# Ajuste dos rótulos do eixo X do barplot para não encavalar
fig.update_xaxes(tickangle=45, row=1, col=2)

fig.show()

#### H3. Lojas com competidores à mais tempo deveriam vendem mais. 

**Falso**: Lojas com COMPETIDORES à mais tempo vendem MENOS.

In [ ]:
aux1 = (
    df4[["competition_time_month", "sales"]].groupby("competition_time_month").sum().reset_index()
)
aux2 = aux1[(aux1["competition_time_month"] < 120) & (aux1["competition_time_month"] != 0)].copy()

corr_matrix = aux2[["competition_time_month", "sales"]].corr(method="pearson")

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Vendas por Meses de Concorrência (Barras)",
        "Tendência de Vendas vs Tempo de Concorrência",
        "Matriz de Correlação (Pearson)",
    ),
)

# Subplot 1: Barplot
fig_bar = px.bar(aux2, x="competition_time_month", y="sales", text_auto=".2s")
for trace in fig_bar.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot 2: Scatterplot com Linha de Tendência (Regressão OLS)
fig_reg = px.scatter(aux2, x="competition_time_month", y="sales", trendline="ols")
for trace in fig_reg.data:
    fig.add_trace(trace, row=1, col=2)

# Subplot 3: Heatmap
fig_heatmap = px.imshow(corr_matrix, text_auto=".2f", color_continuous_scale="RdBu_r")
for trace in fig_heatmap.data:
    fig.add_trace(trace, row=1, col=3)

# Ajustes de Layout
fig.update_layout(
    height=500,
    width=1600,
    template="plotly_white",
    showlegend=False,
    title_text="Análise do Tempo de Concorrência (Meses) vs Vendas",
)

fig.update_xaxes(title_text="Meses de Concorrência", row=1, col=1)
fig.update_xaxes(title_text="Meses de Concorrência", row=1, col=2)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=1)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=2)

fig.show()

#### H4. Lojas com promoções ativas por mais tempo deveriam vender mais.

**Falso**: Lojas com promoções ativas por mais tempo vendem menos, depois de um certo periodo de promoção.

In [ ]:
aux1 = df4[["promo_time_week", "sales"]].groupby("promo_time_week").sum().reset_index()
aux2 = aux1[aux1["promo_time_week"] > 0].copy()
aux3 = aux1[aux1["promo_time_week"] < 0].copy()
corr_matrix = aux1[["promo_time_week", "sales"]].corr(method="pearson")

fig = make_subplots(
    rows=2,
    cols=3,
    specs=[[{}, {}, {"rowspan": 2}], [{}, {}, None]],
    subplot_titles=(
        "Promo Extended (Semanas > 0)",
        "Tendência Promo Extended",
        "Matriz de Correlação",
        "Antes da Promo (Semanas < 0)",
        "Tendência Antes da Promo",
    ),
)

# Subplot (1,1): Barplot Promo Extended (> 0)
fig_bar1 = px.bar(aux2, x="promo_time_week", y="sales", text_auto=".2s")
for trace in fig_bar1.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot (1,2): Regressão Promo Extended (> 0)
fig_reg1 = px.scatter(aux2, x="promo_time_week", y="sales", trendline="ols")
for trace in fig_reg1.data:
    fig.add_trace(trace, row=1, col=2)

# Subplot (2,1): Barplot Antes da Promo (< 0)
fig_bar2 = px.bar(aux3, x="promo_time_week", y="sales", text_auto=".2s")
for trace in fig_bar2.data:
    fig.add_trace(trace, row=2, col=1)

# Subplot (2,2): Regressão Antes da Promo (< 0)
fig_reg2 = px.scatter(aux3, x="promo_time_week", y="sales", trendline="ols")
for trace in fig_reg2.data:
    fig.add_trace(trace, row=2, col=2)

# Subplot (1,3): Heatmap ocupando as 2 linhas da 3ª coluna
fig_heatmap = px.imshow(corr_matrix, text_auto=".2f", color_continuous_scale="RdBu_r")
for trace in fig_heatmap.data:
    fig.add_trace(trace, row=1, col=3)

# Ajustes de Layout
fig.update_layout(
    height=800,
    width=1600,
    template="plotly_white",
    showlegend=False,
    title_text="Análise do Tempo de Promoção Ativa (Semanas) vs Vendas",
)

fig.update_xaxes(tickangle=90, row=1, col=1)
fig.update_xaxes(tickangle=90, row=2, col=1)

fig.show()

#### H6. Lojas com mais promoções consecutivas deveriam vender mais.

**Falsa**: Lojas com mais promoções consecutivas vendem menos.

In [ ]:
df4[["promo", "promo2", "sales"]].groupby(["promo", "promo2"]).sum().reset_index()

In [ ]:
aux1 = (
    df4[(df4["promo"] == 1) & (df4["promo2"] == 1)][["year_week", "sales"]]
    .groupby("year_week")
    .sum()
    .reset_index()
)
ax = aux1.plot()

aux2 = (
    df4[(df4["promo"] == 1) & (df4["promo2"] == 0)][["year_week", "sales"]]
    .groupby("year_week")
    .sum()
    .reset_index()
)
aux2.plot(ax=ax)

ax.legend(labels=["Tradicional & Extendida", "Extendida"]);

#### H7. Lojas abertas durante o feriado de Natal deveriam vender mais.

**Falsa**: Lojas abertas durante o feriado do Natal vendem menos.

In [ ]:
aux = df4[df4["state_holiday"] != "regular_day"]

aux1 = aux[["state_holiday", "sales"]].groupby("state_holiday").sum().reset_index()
aux2 = (
    aux[["year", "state_holiday", "sales"]].groupby(["year", "state_holiday"]).sum().reset_index()
)

# Converte o ano para string para evitar eixo contínuo no Plotly
aux2["year"] = aux2["year"].astype(str)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Total de Vendas por Feriado Estadual",
        "Vendas por Feriado Estadual ao Longo dos Anos",
    ),
)

# Subplot 1: Barplot Total por Feriado
fig_bar1 = px.bar(aux1, x="state_holiday", y="sales", text_auto=".2s")
for trace in fig_bar1.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot 2: Barplot Agrupado por Ano (hue/color='state_holiday')
fig_bar2 = px.bar(
    aux2, x="year", y="sales", color="state_holiday", barmode="group", text_auto=".2s"
)
for trace in fig_bar2.data:
    fig.add_trace(trace, row=1, col=2)

# Ajustes de Layout
fig.update_layout(
    height=500,
    width=1400,
    template="plotly_white",
    title_text="Análise de Vendas em Feriados Estaduais",
    legend={
        "title": "Feriado Estadual",
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "right",
        "x": 1,
    },
)

fig.update_xaxes(title_text="Feriado Estadual", row=1, col=1)
fig.update_xaxes(title_text="Ano", row=1, col=2)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=1)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=2)

fig.show()

#### H8. Lojas deveriam vender mais ao longo dos anos.

**Falsa**: Lojas vendem menos ao longo dos anos.

In [ ]:
aux1 = df4[["year", "sales"]].groupby("year").sum().reset_index()

# Matriz de Correlação
corr_matrix = aux1.corr(method="pearson")

# Converte o ano para string para evitar eixos contínuos com casas decimais nos barplots
aux1_str = aux1.copy()
aux1_str["year"] = aux1_str["year"].astype(str)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Vendas por Ano (Barras)",
        "Tendência de Vendas vs Ano",
        "Matriz de Correlação (Pearson)",
    ),
)

# Subplot 1: Barplot
fig_bar = px.bar(aux1_str, x="year", y="sales", text_auto=".2s")
for trace in fig_bar.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot 2: Scatterplot com Linha de Tendência (Regressão OLS)
fig_reg = px.scatter(aux1, x="year", y="sales", trendline="ols")
for trace in fig_reg.data:
    fig.add_trace(trace, row=1, col=2)

# Subplot 3: Heatmap
fig_heatmap = px.imshow(corr_matrix, text_auto=".2f", color_continuous_scale="RdBu_r")
for trace in fig_heatmap.data:
    fig.add_trace(trace, row=1, col=3)

# Ajustes de Layout
fig.update_layout(
    height=500,
    width=1600,
    template="plotly_white",
    showlegend=False,
    title_text="Análise Anual de Vendas (Year vs Sales)",
)

fig.update_xaxes(title_text="Ano", row=1, col=1)
fig.update_xaxes(title_text="Ano", row=1, col=2)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=1)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=2)

fig.show()

#### H9. Lojas deveriam vender mais no segundo semestre do ano.

**Falsa**: Lojas vendem menos no segundo semestre do ano.

In [ ]:
aux1 = df4[["month", "sales"]].groupby("month").sum().reset_index()

# Matriz de Correlação
corr_matrix = aux1.corr(method="pearson")

# Converte o mês para string para evitar eixos contínuos no Barplot
aux1_str = aux1.copy()
aux1_str["month"] = aux1_str["month"].astype(str)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Vendas por Mês (Barras)",
        "Tendência de Vendas vs Mês",
        "Matriz de Correlação (Pearson)",
    ),
)

# Subplot 1: Barplot
fig_bar = px.bar(aux1_str, x="month", y="sales", text_auto=".2s")
for trace in fig_bar.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot 2: Scatterplot com Linha de Tendência (Regressão OLS)
fig_reg = px.scatter(aux1, x="month", y="sales", trendline="ols")
for trace in fig_reg.data:
    fig.add_trace(trace, row=1, col=2)

# Subplot 3: Heatmap
fig_heatmap = px.imshow(corr_matrix, text_auto=".2f", color_continuous_scale="RdBu_r")
for trace in fig_heatmap.data:
    fig.add_trace(trace, row=1, col=3)

# Ajustes de Layout
fig.update_layout(
    height=500,
    width=1600,
    template="plotly_white",
    showlegend=False,
    title_text="Análise Mensal de Vendas (Month vs Sales)",
)

fig.update_xaxes(title_text="Mês", row=1, col=1)
fig.update_xaxes(title_text="Mês", row=1, col=2)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=1)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=2)

fig.show()

#### H10. Lojas deveriam vender mais depois do dia 10 de cada mês.

**Verdade**: Lojas vendem mais depois do dia 10 de cada mês.

In [ ]:
aux1 = df4[["day", "sales"]].groupby("day").sum().reset_index()

# Matriz de Correlação
corr_matrix = aux1.corr(method="pearson")

# Converte o dia para string para evitar eixos contínuos no Barplot
aux1_str = aux1.copy()
aux1_str["day"] = aux1_str["day"].astype(str)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Vendas por Dia do Mês (Barras)",
        "Tendência de Vendas vs Dia do Mês",
        "Matriz de Correlação (Pearson)",
    ),
)

# Subplot 1: Barplot
fig_bar = px.bar(aux1_str, x="day", y="sales", text_auto=".2s")
for trace in fig_bar.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot 2: Scatterplot com Linha de Tendência (Regressão OLS)
fig_reg = px.scatter(aux1, x="day", y="sales", trendline="ols")
for trace in fig_reg.data:
    fig.add_trace(trace, row=1, col=2)

# Subplot 3: Heatmap
fig_heatmap = px.imshow(corr_matrix, text_auto=".2f", color_continuous_scale="RdBu_r")
for trace in fig_heatmap.data:
    fig.add_trace(trace, row=1, col=3)

# Ajustes de Layout
fig.update_layout(
    height=500,
    width=1600,
    template="plotly_white",
    showlegend=False,
    title_text="Análise por Dia do Mês de Vendas (Day vs Sales)",
)

fig.update_xaxes(title_text="Dia do Mês", row=1, col=1)
fig.update_xaxes(title_text="Dia do Mês", row=1, col=2)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=1)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=2)

fig.show()

In [ ]:
aux1["before_after"] = aux1["day"].apply(lambda x: "before_10_days" if x <= 10 else "after_10_days")
aux2 = aux1[["before_after", "sales"]].groupby("before_after").sum().reset_index()

# Gráfico de Barras em Plotly
fig = px.bar(
    aux2,
    x="before_after",
    y="sales",
    text_auto=".2s",
    title="Total de Vendas: Antes vs Depois dos Primeiros 10 Dias do Mês",
    template="plotly_white",
)

# Ajustes de Layout
fig.update_layout(height=500, width=700, xaxis_title="Período", yaxis_title="Total de Vendas")

fig.show()

### H11. Lojas deveriam vender menos aos finais de semana.

**Verdade**: Lojas vendem menos nos finais de semana.

In [ ]:
aux1 = df4[["day_of_week", "sales"]].groupby("day_of_week").sum().reset_index()

# Matriz de Correlação
corr_matrix = aux1.corr(method="pearson")

# Converte o dia da semana para string para evitar eixos contínuos no Barplot
aux1_str = aux1.copy()
aux1_str["day_of_week"] = aux1_str["day_of_week"].astype(str)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Vendas por Dia da Semana (Barras)",
        "Tendência de Vendas vs Dia da Semana",
        "Matriz de Correlação (Pearson)",
    ),
)

# Subplot 1: Barplot
fig_bar = px.bar(aux1_str, x="day_of_week", y="sales", text_auto=".2s")
for trace in fig_bar.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot 2: Scatterplot com Linha de Tendência (Regressão OLS)
fig_reg = px.scatter(aux1, x="day_of_week", y="sales", trendline="ols")
for trace in fig_reg.data:
    fig.add_trace(trace, row=1, col=2)

# Subplot 3: Heatmap
fig_heatmap = px.imshow(corr_matrix, text_auto=".2f", color_continuous_scale="RdBu_r")
for trace in fig_heatmap.data:
    fig.add_trace(trace, row=1, col=3)

# Ajustes de Layout
fig.update_layout(
    height=500,
    width=1600,
    template="plotly_white",
    showlegend=False,
    title_text="Análise por Dia da Semana de Vendas (Day of Week vs Sales)",
)

fig.update_xaxes(title_text="Dia da Semana", row=1, col=1)
fig.update_xaxes(title_text="Dia da Semana", row=1, col=2)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=1)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=2)

fig.show()

#### H12. Lojas deveriam vender menos durante os feriados escolares.

**Verdade**: Lojas vendem menos durante os feriados escolares, exceto os meses de Julho e Agosto.

In [ ]:
aux1 = df4[["school_holiday", "sales"]].groupby("school_holiday").sum().reset_index()
aux2 = (
    df4[["month", "school_holiday", "sales"]]
    .groupby(["month", "school_holiday"])
    .sum()
    .reset_index()
)

aux1["school_holiday"] = aux1["school_holiday"].astype(str)
aux2["school_holiday"] = aux2["school_holiday"].astype(str)
aux2["month"] = aux2["month"].astype(str)

fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("Total de Vendas por Feriado Escolar", "Vendas Mensais por Feriado Escolar"),
)

# Subplot 1: Barplot Total
fig_bar1 = px.bar(aux1, x="school_holiday", y="sales", text_auto=".2s")
for trace in fig_bar1.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot 2: Barplot Agrupado por Mês e Feriado (hue/color='school_holiday')
fig_bar2 = px.bar(
    aux2, x="month", y="sales", color="school_holiday", barmode="group", text_auto=".2s"
)
for trace in fig_bar2.data:
    fig.add_trace(trace, row=2, col=1)

# Ajustes de Layout
fig.update_layout(
    height=800,
    width=1200,
    template="plotly_white",
    title_text="Análise de Vendas em Feriados Escolares (School Holiday)",
    legend={
        "title": "Feriado Escolar",
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "right",
        "x": 1,
    },
)

fig.update_xaxes(title_text="Feriado Escolar (0 = Não, 1 = Sim)", row=1, col=1)
fig.update_xaxes(title_text="Mês", row=2, col=1)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=1)
fig.update_yaxes(title_text="Total de Vendas", row=2, col=1)

fig.show()

In [ ]:
day_map = {
    1: "Segunda",
    2: "Terça",
    3: "Quarta",
    4: "Quinta",
    5: "Sexta",
    6: "Sábado",
    7: "Domingo",
}

aux1 = df4[["day_of_week", "sales"]].groupby("day_of_week").sum().reset_index()
aux1_str = aux1.copy()
aux1_str["day_name"] = aux1_str["day_of_week"].map(day_map)

# Matriz de correlação com valores numéricos
corr_matrix = aux1[["day_of_week", "sales"]].corr(method="pearson")

# Grid 1x3
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Vendas por Dia da Semana (Barras)",
        "Tendência de Vendas vs Dia da Semana",
        "Matriz de Correlação (Pearson)",
    ),
)

# Subplot 1: Barplot (com nomes dos dias)
fig_bar = px.bar(aux1_str, x="day_name", y="sales", text_auto=".2s")
for trace in fig_bar.data:
    fig.add_trace(trace, row=1, col=1)

# Subplot 2: Regressão OLS (usando números 1 a 7)
fig_reg = px.scatter(aux1, x="day_of_week", y="sales", trendline="ols")
for trace in fig_reg.data:
    fig.add_trace(trace, row=1, col=2)

# Subplot 3: Heatmap (annot=True do seaborn equivale a text_auto em Plotly)
fig_heatmap = px.imshow(corr_matrix, text_auto=".2f", color_continuous_scale="RdBu_r")
for trace in fig_heatmap.data:
    fig.add_trace(trace, row=1, col=3)

# Ajustes de Layout
fig.update_layout(
    height=500,
    width=1600,
    template="plotly_white",
    showlegend=False,
    title_text="Análise de Vendas por Dia da Semana (Segunda = 1 a Domingo = 7)",
)

fig.update_xaxes(title_text="Dia da Semana", row=1, col=1)
fig.update_xaxes(title_text="Dia da Semana (1-7)", row=1, col=2)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=1)
fig.update_yaxes(title_text="Total de Vendas", row=1, col=2)

fig.show()

### 4.2.1 Resumo das Hipoteses

In [ ]:
tab = [
    ["Hipoteses", "Conclusão", "Relevancia"],
    ["H1", "Falsa", "Baixa"],
    ["H2", "Falsa", "Media"],
    ["H3", "Falsa", "Media"],
    ["H4", "Falsa", "Baixa"],
    ["H5", "-", "-"],
    ["H6", "Falsa", "Baixa"],
    ["H7", "Falsa", "Media"],
    ["H8", "Falsa", "Alta"],
    ["H9", "Falsa", "Alta"],
    ["H10", "Verdadeira", "Alta"],
    ["H11", "Verdadeira", "Alta"],
    ["H12", "Verdadeira", "Baixa"],
]

print(tabulate(tab, headers="firstrow"))

## 4.3 - Analise Multivariada

### 4.3.1 Atributos Numericos

In [ ]:
correlation = num_attributes.corr(method="pearson")

fig = px.imshow(
    correlation,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Matriz de Correlação (Pearson)",
)

# Ajustes de Layout
fig.update_layout(height=700, width=900, template="plotly_white")

fig.show()

### 4.3.2 Atributos Categoricos

In [ ]:
a = df4.select_dtypes(include="object")

a1 = cramer_v(a["state_holiday"], a["state_holiday"])
a2 = cramer_v(a["state_holiday"], a["store_type"])
a3 = cramer_v(a["state_holiday"], a["assortment"])

a4 = cramer_v(a["store_type"], a["state_holiday"])
a5 = cramer_v(a["store_type"], a["store_type"])
a6 = cramer_v(a["store_type"], a["assortment"])

a7 = cramer_v(a["assortment"], a["state_holiday"])
a8 = cramer_v(a["assortment"], a["store_type"])
a9 = cramer_v(a["assortment"], a["assortment"])

d = pd.DataFrame(
    {"state_holiday": [a1, a2, a3], "store_types": [a4, a5, a6], "assortment": [a7, a8, a9]}
)

d = d.set_index(d.columns)

In [ ]:
fig = px.imshow(
    d,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Matriz de Correlação / Associação (Cramér's V)",
)

# Ajustes de Layout
fig.update_layout(height=700, width=900, template="plotly_white")

fig.show()

# 5.0 Passo 05 - Preparação dos Dados

In [ ]:
df5 = df4.copy()

## 5.1 Normalização

## 5.2 Redimensionamento

In [ ]:
rs = RobustScaler()

df5["competition_distance"] = rs.fit_transform(df5[["competition_distance"]].values)
df5["year"] = rs.fit_transform(df5[["year"]].values)
df5["competition_time_month"] = rs.fit_transform(df5[["competition_time_month"]].values)
df5["promo_time_week"] = rs.fit_transform(df5[["promo_time_week"]].values)

## 5.3 Transformação

### 5.3.1 Codificação

In [ ]:
# Aplicando One Hot Encoding
df5 = pd.get_dummies(df5, prefix=["state_holiday"], columns=["state_holiday"])

In [ ]:
# Aplicando Label Encoding
le = LabelEncoder()
df5["store_type"] = le.fit_transform(df5["store_type"])

In [ ]:
# Aplicando Ordinal Enconding
assortment_dict = {"basic": 1, "extra": 2, "extended": 3}
df5["assortment"] = df5["assortment"].map(assortment_dict)

### 5.3.2 Transformação Da Variável Resposta

In [ ]:
# Aplicando Transformação Logaritma
df5["sales"] = np.log1p(df5["sales"])

In [ ]:
# Aplicando Transformação de Natureza Cíclica
df5["day_of_week_sin"] = df5["day_of_week"].apply(lambda x: np.sin(x * (2.0 * np.pi / 7)))
df5["day_of_week_cos"] = df5["day_of_week"].apply(lambda x: np.cos(x * (2.0 * np.pi / 7)))

df5["month_sin"] = df5["month"].apply(lambda x: np.sin(x * (2.0 * np.pi / 12)))
df5["month_cos"] = df5["month"].apply(lambda x: np.cos(x * (2.0 * np.pi / 12)))

df5["day_sin"] = df5["day"].apply(lambda x: np.sin(x * (2.0 * np.pi / 30)))
df5["day_cos"] = df5["day"].apply(lambda x: np.cos(x * (2.0 * np.pi / 30)))

df5["week_of_year_sin"] = df5["week_of_year"].apply(lambda x: np.sin(x * (2.0 * np.pi / 52)))
df5["week_of_year_cos"] = df5["week_of_year"].apply(lambda x: np.cos(x * (2.0 * np.pi / 52)))

# 6.0 Passo 06 - Seleção De Variáveis

In [ ]:
df6 = df5.copy()

## 6.1 Dividindo O DF Em Treino E Teste

In [ ]:
cols_drop = ["week_of_year", "day", "month", "day_of_week", "competition_since", "year_week"]
df6 = df6.drop(cols_drop, axis=1)

In [ ]:
# Dataset de Treino
X_train = df6[df6["date"] < "2015-06-19"]
y_train = X_train["sales"]

# Dataset de Teste
X_test = df6[df6["date"] >= "2015-06-19"]
y_test = X_test["sales"]

print(f"Treino Data minima: {X_train['date'].min()}")
print(f"Treino Data maxima: {X_train['date'].max()}")

print(f"\nTeste Data minima: {X_test['date'].min()}")
print(f"Teste Data maxima: {X_test['date'].max()}")

## 6.2 Boruta Como Seletor de Variáveis

In [ ]:
# Treino e teste do dataset para o Boruta
X_train_n = X_train.drop(["date", "sales", "promo2_since"], axis=1)
y_train_n = y_train.values.ravel()

features_names = X_train_n.columns.to_list()

# Preparando os tipos das colunas de valores para boruta aceitar
# Converte booleanos para int (0 ou 1) para evitar outro erro de tipo
bool_cols = X_train_n.select_dtypes(include="bool").columns
X_train_n[bool_cols] = X_train_n[bool_cols].astype(int)

# Converte o DataFrame para Array NumPy (Requisito do BorutaPy)
X_train_array = X_train_n.values

# Defino RandomForestRegressor
rfr = RandomForestRegressor(n_jobs=-1)

# Defino Boruta
boruta = BorutaPy(rfr, n_estimators="auto", verbose=2, random_state=42).fit(
    X_train_array, y_train_n
)

### 6.2.1 Melhores Variaveis Pelo Boruta

In [ ]:
cols_selected = boruta.support_.tolist()

# Melhores variaveis
X_train_fs = X_train.drop(["date", "sales"], axis=1)
cols_selected_boruta = X_train_fs.iloc[:, cols_selected].columns.to_list()

# Os não selecionados
cols_not_selected_boruta = list(np.setdiff1d(X_train_fs.columns, cols_selected_boruta))

## 6.3 Seleção Manual De Variaveis

In [ ]:
cols_selected_boruta = [
    "store",
    "promo",
    "store_type",
    "assortment",
    "competition_distance",
    "competition_open_since_month",
    "competition_open_since_year",
    "promo2",
    "promo2_since_week",
    "promo2_since_year",
    "competition_time_month",
    "promo_time_week",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos",
    "week_of_year_sin",
    "week_of_year_cos",
]

# Colunas para adicionar
feat_to_add = ["date", "sales"]

# resultado final
cols_selected_boruta.extend(feat_to_add)

# 7.0 Passo 07 - Modelando Machine Learning

In [ ]:
x_train = X_train[cols_selected_boruta]
x_test = X_test[cols_selected_boruta]

In [ ]:
# Garante que a transformação seja feita no x_train e x_test que o modelo usa
min_date = x_train["date"].min()

x_train["date"] = (x_train["date"] - min_date).dt.days
x_test["date"] = (x_test["date"] - min_date).dt.days

## 7.1 Modelo Médio

In [ ]:
aux1 = x_test.copy()
aux1["sales"] = y_test.copy()

# Predição
aux2 = (
    aux1[["store", "sales"]]
    .groupby("store")
    .mean()
    .reset_index()
    .rename(columns={"sales": "predictions"})
)
aux1 = pd.merge(aux1, aux2, how="left", on="store")
yhat_baseline = aux1["predictions"]

# Performace
baseline_result = ml_error_regression("Average Model", np.expm1(y_test), np.expm1(yhat_baseline))
baseline_result

## 7.2 Linear Regression

In [ ]:
# Modelo
lr = LinearRegression().fit(x_train, y_train)

# Predição
yhat_lr = lr.predict(x_test)

# Performance
lr_result = ml_error_regression("Linear Regression", np.expm1(y_test), np.expm1(yhat_lr))
lr_result

### 7.2.1 Linear Regression - Cross Validation

In [ ]:
lr_result_cv = cross_validation(x_train, 5, "Linenar Regression", lr)
lr_result_cv

## 7.3 Linear Regression Regularized - Lasso

In [ ]:
# Modelo
lrr = Lasso(alpha=0.01).fit(x_train, y_train)

# Predição
yhat_lrr = lrr.predict(x_test)

# Performance
lrr_result = ml_error_regression(
    "Linear Regression Regularized", np.expm1(y_test), np.expm1(yhat_lrr)
)
lrr_result

### 7.3.1 Linear Regression Regularizes - Lasso - Cross Validation

In [ ]:
lrr_result_cv = cross_validation(x_train, 5, "Linenar Regression Regularizes - Lasso", lrr)
lrr_result_cv

## 7.4 Random Forest Regressor

In [ ]:
# Modelo
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42).fit(x_train, y_train)

# Predição
yhat_rf = rf.predict(x_test)

# Performance
rf_result = ml_error_regression("Random Forest Regressor", np.expm1(y_test), np.expm1(yhat_rf))
rf_result

### 7.4.1 Random Forest Regressor - Cross Validation

In [ ]:
rf_result_cv = cross_validation(x_train, 5, "Random Forest Regressor", rf)
rf_result_cv

## 7.5 XGBoost Regressor

In [ ]:
# Modelo
model_xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=100,
    learning_rate=0.01,
    max_depth=10,
    subsample=0.7,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
).fit(x_train, y_train)

# Predição
yhat_xgb = model_xgb.predict(x_test)

# Performance
xgb_result = ml_error_regression("XGBoost Regressor", np.expm1(y_test), np.expm1(yhat_xgb))
xgb_result

### 7.5.1 XGBoost Regressor - Cross Validation

In [ ]:
xgb_result_cv = cross_validation(x_train, 5, "XGBoost Regressor", model_xgb)
xgb_result_cv

## 7.6 Comparando Performance De Modelos

In [ ]:
modelling_result = pd.concat([baseline_result, lr_result, lrr_result, rf_result, xgb_result])
modelling_result.sort_values("RMSE")

In [ ]:
modelling_result_cv = pd.concat([lr_result_cv, lrr_result_cv, rf_result_cv, xgb_result_cv])
modelling_result_cv.sort_values("RMSE CV")

# 8.0 Passo 8 - Ajuste Fino de Hiperparâmetros

## 8.1 Random Search

In [ ]:
param = {
    "n_estimators": [15, 17, 25, 30, 35],
    "learning_rate": [0.01, 0.03],
    "max_depth": [3, 5, 9],
    "subsample": [0.1, 0.5, 0.7],
    "colsample_bytree": [0.3, 0.7, 0.9],
    "min_child_weight": [3, 8, 15],
}

MAX_EVAL = 100

In [ ]:
final_result = pd.DataFrame()

for i in range(MAX_EVAL):
    print(f"Teste Numero: {i}")
    hp = {k: sample(v, 1)[0] for k, v in param.items()}
    print(hp)
    print("\n")

    # Modelo
    model_xgb = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=hp["n_estimators"],
        learning_rate=hp["learning_rate"],
        max_depth=hp["max_depth"],
        subsample=hp["subsample"],
        colsample_bytree=hp["colsample_bytree"],
        min_child_weight=hp["min_child_weight"],
        random_state=42,
        n_jobs=-1,
    )

    # Performance
    result = cross_validation(x_train, 5, "XGBoost Regression", model_xgb)
    result.insert(0, "iter", i)
    final_result = pd.concat([final_result, result], ignore_index=True)

# Extrai apenas a primeira parte numérica antes do símbolo de '+'
final_result["MAE_num"] = final_result["MAE CV"].str.split("+").str[0].astype(float)

# Ordena do menor para o maior MAE
final_result_sorted = final_result.sort_values(by="MAE_num", ascending=True).drop(
    columns=["MAE_num"]
)

final_result_sorted

## 8.2 Modelo Final

In [ ]:
param_tuned = {
    "n_estimators": 35,
    "learning_rate": 0.03,
    "max_depth": 9,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "min_child_weight": 8,
}

In [ ]:
# Modelo
model_xgb_tuned = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=param_tuned["n_estimators"],
    learning_rate=param_tuned["learning_rate"],
    max_depth=param_tuned["max_depth"],
    subsample=param_tuned["subsample"],
    colsample_bytree=param_tuned["colsample_bytree"],
    min_child_weight=param_tuned["min_child_weight"],
    random_state=42,
    n_jobs=-1,
).fit(x_train, y_train)

# Predição
yhat_xgb_tuned = model_xgb_tuned.predict(x_test)

# Performance
xgb_result_tuned = ml_error_regression(
    "XGBoost Regressor", np.expm1(y_test), np.expm1(yhat_xgb_tuned)
)
xgb_result_tuned

## 8.3 Guardando O Melhor Modelo

In [ ]:
model_path = r"models/model_rossmann.pkl"

with Path.open(model_path, "wb") as file:
    dump(model_xgb_tuned, file)